This script is responsible for training the Models on the given dataset and save the best metadata.

1. Set Up Environment

In [ ]:
# Install required packages
!pip install aim pyngrok albumentations schedulefree torchinfo torchmetrics

In [ ]:
!pip install git+https://github.com/qubvel/segmentation_models.pytorch

# --- 1. Imports ---

In [ ]:
# --- 1. Imports ---
print("Importing libraries...")
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import schedulefree
import zipfile
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError
import gc
import time
from tqdm import tqdm
import torchvision.models as models
from datetime import datetime
import torch.nn.functional as F
from google.colab import userdata
import random
import seaborn as sns
import shutil
import smtplib
from email.mime.text import MIMEText
from torch.cuda.amp import autocast, GradScaler
import cv2
import segmentation_models_pytorch as smp
import numpy as np
import albumentations as A
from albumentations.pytorch import ToTensorV2
import warnings
from sklearn.metrics import auc as sklearn_auc
import json
from sklearn.metrics import roc_auc_score
from torchvision import transforms
from torchvision.transforms import functional as TF
from sklearn.metrics import confusion_matrix, accuracy_score
from PIL import Image, UnidentifiedImageError
import torchvision.models as models
import re
from collections import defaultdict
from torchmetrics import PrecisionRecallCurve, AUROC, AveragePrecision
from torchmetrics.classification import BinaryAveragePrecision, BinaryAUROC
import math
from collections import defaultdict
import contextlib
import sys
from torch.utils.data import WeightedRandomSampler


print("Libraries imported.")

In [ ]:
# --- 5. Configuration & Setup ---
print("Configuring environment...")
# --- Device Configuration ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

IS_DPT = False

# --- Hyperparameters ---
BATCH_SIZE = 32 # Slightly reduced default batch size, adjust based on GPU memory
NUM_EPOCHS = 100
WORKERS = 2 # Keep > 0 for persistent workers
PATIENCE = 5 # Increased patience slightly
UNLEASHED = False # Set to True to disable early stopping

# --- Add SMP specific config ---
ENCODER_WEIGHTS = "imagenet" # Use pre-trained weights
ACTIVATION = None # Output raw logits for BCEWithLogitsLoss/Hybrid loss

# --- Paths and Labels ---
CHECKPOINT_PATH = 'Checkpoints' # place to save training

# --- Email Settings ---
sender = "bruno.nunes.1987@gmail.com"
recipients = ["bruno.nunes.1987@gmail.com"]
password = userdata.get('APP_PASSWORD') # Ensure this secret exists in Colab

# --- Aim Repository Path ---
AIM_REPO_PATH = "aim_repo_prostate"

# --- Dataset Source Path ---
DATASET_ZIP_DIR = 'IA_MEDICA_SAMPLES/DIAGSET/REINHARD'
METADATA_DIR = 'METADATA_CHECKPOINTS/DIAGSET/REINHARD'
IDENTIFIER = 'DIAGSET_REINHARD

# --- Data Extraction Directory ---
base_data_dir = '/content/dataset' # Extracted fold data goes here

# --- Loss Weights (from paper) ---
# These are now only used for the BCEDiceHybrid loss
ALPHA_BCE = 0.125
BETA_DICE_BG = 0.157
GAMMA_DICE_FG = 0.173

#replication
SEED = 24

# Use a consistent, reasonable LR unless you have strong fold-specific reasons
ENCODER_LR_FACTOR = 0.1 # Encoder LR will be BASE_LEARNING_RATE * ENCODER_LR_FACTOR
DECODER_DROPOUT = 0.3 # Define your desired dropout rate

# Set to True to use a smaller, stratified subset of the data for faster training.
USE_SUBSET = True
# The ratio of the full dataset to use (e.g., 0.10 = 10%).
SUBSET_RATIO = 0.25

OPTIMIZER_NAME = "AdamWScheduleFree" # options are AdamWScheduleFree, AdamW

USE_TTA_FOR_EVAL = True

SENSITIVITY_TARGET = 0.95 # ✔ design choice

SIMULATION_ONLY = False

OVERSAMPLE_FACTOR =

STATS_SAMPLE_SIZE = None #20000
MEAN = [0.5631743144356475, 0.3779451521216607, 0.6975475908629748]
STD = [0.07002276986060542, 0.05347828888148516, 0.047768733057653855]

In [ ]:
def get_learning_rate(architecture: str):
    CONFIGS = {
        "SWIN":       {"lr": 3e-4, "wd": 1e-4},
        "SEGFORMER":  {"lr": 3e-4, "wd": 1e-4},
        "DPT":        {"lr": 3e-5, "wd": 1e-4},
        "UPERNET":    {"lr": 2e-4, "wd": 1e-4},
        "DEEPLABV3PLUS": {"lr": 3e-4, "wd": 1e-4},
        "UNET++":        {"lr": 3e-3, "wd": 1e-4},
        "FPN":           {"lr": 3e-4, "wd": 1e-4},
        "MANET":         {"lr": 5e-3, "wd": 1e-4},
    }

    try:
        cfg = CONFIGS[architecture.upper()]
    except KeyError:
        raise ValueError(f"Unknown architecture: {architecture}")

    return cfg["lr"], cfg["wd"]

In [ ]:
def autocast_ctx(x: torch.Tensor, amp_dtype):
    use_cuda_amp = x.is_cuda and (amp_dtype is not torch.float32)
    return torch.amp.autocast("cuda", dtype=amp_dtype) if use_cuda_amp else contextlib.nullcontext()

In [ ]:
def change_paths(path, possible_paths=None):
    if possible_paths is None:
        possible_paths = ['/content/drive/MyDrive/Personal_Drive_Bruno/','/content/drive/MyDrive/']
    for p in possible_paths:
      path=os.path.join(p,path)
      if os.path.exists(path):
        return path
    raise ValueError(f"The path {path} does not match any known environments.")

In [ ]:
CHECKPOINT_PATH=change_paths(CHECKPOINT_PATH)
AIM_REPO_PATH=change_paths(AIM_REPO_PATH)
DATASET_ZIP_DIR=change_paths(DATASET_ZIP_DIR)
METADATA_DIR=change_paths(METADATA_DIR)

In [ ]:
# --- Paths within the extracted fold ---
train_cancer_image_dir = os.path.join(base_data_dir, 'TRAIN/CANCER')
train_cancer_mask_dir = os.path.join(base_data_dir, 'TRAIN/CANCER_MASK')
train_not_cancer_image_dir = os.path.join(base_data_dir, 'TRAIN/NOT_CANCER')
train_not_cancer_mask_dir = os.path.join(base_data_dir, 'TRAIN/NOT_CANCER_MASK')
val_cancer_image_dir = os.path.join(base_data_dir, 'VALIDATION/CANCER')
val_cancer_mask_dir = os.path.join(base_data_dir, 'VALIDATION/CANCER_MASK')
val_not_cancer_image_dir = os.path.join(base_data_dir, 'VALIDATION/NOT_CANCER')
val_not_cancer_mask_dir = os.path.join(base_data_dir, 'VALIDATION/NOT_CANCER_MASK')

In [ ]:
class TrainingHealthTracker:
    """
    Tracks training/validation health issues:
    - NaN/Inf loss counts
    - Skipped batch counts (with reasons)
    - Validation collapse count (v_results is None)
    - Invalid metric counts (AUPRC/AUROC/MCC* NaN/Inf)
    """

    def __init__(self, name="run"):
        self.name = name
        self.reset_run()
        self.reset_epoch()

    def reset_run(self):
        self.run = {
            "train_naninf_loss": 0,
            "val_naninf_loss": 0,
            "train_skipped_batches": 0,
            "val_skipped_batches": 0,
            "train_skip_reasons": defaultdict(int),
            "val_skip_reasons": defaultdict(int),
            "val_collapse_epochs": 0,
            "val_invalid_metric_epochs": 0,
        }

    def reset_epoch(self):
        self.epoch = {
            "train_naninf_loss": 0,
            "val_naninf_loss": 0,
            "train_skipped_batches": 0,
            "val_skipped_batches": 0,
            "train_skip_reasons": defaultdict(int),
            "val_skip_reasons": defaultdict(int),
            "val_collapsed": 0,           # 0/1 for this epoch
            "val_invalid_metrics": 0,     # 0/1 for this epoch
        }

    # ---------------------------
    # Logging helpers
    # ---------------------------
    def train_skip(self, reason: str):
        self.epoch["train_skipped_batches"] += 1
        self.epoch["train_skip_reasons"][reason] += 1
        self.run["train_skipped_batches"] += 1
        self.run["train_skip_reasons"][reason] += 1

    def val_skip(self, reason: str):
        self.epoch["val_skipped_batches"] += 1
        self.epoch["val_skip_reasons"][reason] += 1
        self.run["val_skipped_batches"] += 1
        self.run["val_skip_reasons"][reason] += 1

    def train_naninf_loss(self):
        self.epoch["train_naninf_loss"] += 1
        self.run["train_naninf_loss"] += 1

    def val_naninf_loss(self):
        self.epoch["val_naninf_loss"] += 1
        self.run["val_naninf_loss"] += 1

    def mark_val_collapsed(self):
        self.epoch["val_collapsed"] = 1
        self.run["val_collapse_epochs"] += 1

    def mark_val_invalid_metrics(self):
        self.epoch["val_invalid_metrics"] = 1
        self.run["val_invalid_metric_epochs"] += 1

    # ---------------------------
    # Pretty printing
    # ---------------------------
    @staticmethod
    def _fmt_reasons(reason_dict):
        if not reason_dict:
            return "-"
        items = sorted(reason_dict.items(), key=lambda x: (-x[1], x[0]))
        return ", ".join([f"{k}:{v}" for k, v in items])

    def log_epoch(self, epoch_num: int, prefix="[Health]"):
        print(
            f"{prefix} Epoch {epoch_num} | "
            f"Train skip={self.epoch['train_skipped_batches']} "
            f"(reasons: {self._fmt_reasons(self.epoch['train_skip_reasons'])}) | "
            f"Train NaN/Inf loss={self.epoch['train_naninf_loss']} | "
            f"Val skip={self.epoch['val_skipped_batches']} "
            f"(reasons: {self._fmt_reasons(self.epoch['val_skip_reasons'])}) | "
            f"Val NaN/Inf loss={self.epoch['val_naninf_loss']} | "
            f"Val collapsed={self.epoch['val_collapsed']} | "
            f"Val invalid_metrics={self.epoch['val_invalid_metrics']}"
        )

    def log_run(self, prefix="[Health-SUMMARY]"):
        print(
            f"{prefix} Run totals | "
            f"Train skip={self.run['train_skipped_batches']} "
            f"(reasons: {self._fmt_reasons(self.run['train_skip_reasons'])}) | "
            f"Train NaN/Inf loss={self.run['train_naninf_loss']} | "
            f"Val skip={self.run['val_skipped_batches']} "
            f"(reasons: {self._fmt_reasons(self.run['val_skip_reasons'])}) | "
            f"Val NaN/Inf loss={self.run['val_naninf_loss']} | "
            f"Val collapse_epochs={self.run['val_collapse_epochs']} | "
            f"Val invalid_metric_epochs={self.run['val_invalid_metric_epochs']}"
        )

In [ ]:
class BCEDiceHybridLossPaper(nn.Module):
    """
    Paper-faithful implementation of the hybrid loss from:

    Khened, M., Kori, A., Rajkumar, H. et al.
    A generalized deep learning framework for whole-slide image segmentation and analysis.
    Scientific Reports 11, 11579 (2021).
    https://doi.org/10.1038/s41598-021-90444-8

    Loss = alpha * CE + beta * Dice_BG + gamma * Dice_FG

    - CE is binary cross-entropy on the tumor posterior p_i
    - Dice uses squared denominator: sum(p^2) + sum(g^2)
    """

    def __init__(self,
                 alpha: float = 0.5,
                 beta: float = 0.25,
                 gamma: float = 0.25,
                 smooth: float = 1e-6):
        super().__init__()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.smooth = smooth

    @staticmethod
    def _flatten(x):
        # [B,H,W] -> [B,N]
        return x.reshape(x.size(0), -1)

    def _dice_loss_khened(self, p, g):
        """
        p, g: [B,H,W] probabilities and binary GT for ONE class
        Implements:
            DL = 1 - (2 sum(p g)) / (sum(p^2) + sum(g^2))
        """
        p = self._flatten(p)
        g = self._flatten(g)

        intersection = (p * g).sum(dim=1)
        denom = (p.pow(2).sum(dim=1) + g.pow(2).sum(dim=1))

        dice = (2.0 * intersection + self.smooth) / (denom + self.smooth)
        return 1.0 - dice.mean()

    def forward(self, logits, target_one_hot):
        """
        logits:          [B,2,H,W] raw model outputs
        target_one_hot:  [B,2,H,W] one-hot encoded masks
        """

        if logits.shape != target_one_hot.shape:
            raise ValueError(
                f"Shape mismatch: logits {logits.shape}, target {target_one_hot.shape}"
            )

        # -------------------------------------------------
        # 1) Posterior probabilities (softmax)
        # -------------------------------------------------
        probs = torch.softmax(logits, dim=1)

        # Tumor posterior p_i and GT g_i (paper notation)
        p_fg = probs[:, 1, :, :]
        g_fg = target_one_hot[:, 1, :, :]

        p_fg = p_fg.clamp(1e-7, 1.0 - 1e-7)

        # -------------------------------------------------
        # 2) Binary Cross-Entropy (Eq. 2)
        # -------------------------------------------------
        ce_loss = -(g_fg * torch.log(p_fg) +
                    (1.0 - g_fg) * torch.log(1.0 - p_fg))
        ce_loss = ce_loss.mean()

        # -------------------------------------------------
        # 3) Dice losses (Eq. 1)
        # -------------------------------------------------
        # Background
        dice_bg = self._dice_loss_khened(
            probs[:, 0, :, :],
            target_one_hot[:, 0, :, :]
        )

        # Foreground (tumor)
        dice_fg = self._dice_loss_khened(
            probs[:, 1, :, :],
            target_one_hot[:, 1, :, :]
        )

        # -------------------------------------------------
        # 4) Hybrid combination (Eq. 3)
        # -------------------------------------------------
        loss = (
            self.alpha * ce_loss +
            self.beta  * dice_bg +
            self.gamma * dice_fg
        )

        return loss

In [ ]:
class AdvancedMetricTracker:
    """
    Memory-safe, incremental validation tracker for binary segmentation.

    - AUPRC / AUROC computed incrementally using binned thresholds (no full-tensor storage).
    - Collapse detection via FG prevalence @ 0.5.
    - MCC* computed incrementally over a fixed threshold grid [0.1..0.9].

    Notes:
    - Using torchmetrics with `thresholds=int` makes AUPRC/AUROC approximate (binned),
      but memory-safe and usually extremely close with sufficient bins.
    """

    def __init__(
        self,
        device: torch.device,
        metric_bins: int = 4096,  # increase for more precision, still memory-safe
        mcc_thresholds=None,       # defaults to 0.1..0.9 step 0.1
        collapse_low: float = 0.001,   # 0.1%  (as fraction)
        collapse_high: float = 0.99,   # 99%   (as fraction)
        from_logits: bool = True,
    ):
        self.device = device
        self.from_logits = from_logits

        # Torchmetrics streaming (binned) metrics
        self.auprc = BinaryAveragePrecision(thresholds=metric_bins).to(device)
        self.auroc = BinaryAUROC(thresholds=metric_bins).to(device)

        # MCC* thresholds
        if mcc_thresholds is None:
            mcc_thresholds = torch.arange(0.1, 1.0, 0.1)  # 0.1..0.9
        self.mcc_thresholds = torch.as_tensor(mcc_thresholds, dtype=torch.float32)

        self.collapse_low = float(collapse_low)
        self.collapse_high = float(collapse_high)

        self.reset()

    def reset(self):
        # Reset torchmetrics internal state
        self.auprc.reset()
        self.auroc.reset()

        # FG prevalence @ 0.5 stats
        self._pred_pos_at_05 = 0
        self._total_pixels = 0

        # MCC* confusion counts per-threshold (CPU int64 for safety)
        K = int(self.mcc_thresholds.numel())
        self._tp = torch.zeros(K, dtype=torch.int64)
        self._fp = torch.zeros(K, dtype=torch.int64)
        self._tn = torch.zeros(K, dtype=torch.int64)
        self._fn = torch.zeros(K, dtype=torch.int64)

    @staticmethod
    def _extract_probs_fg(pred_logits: torch.Tensor) -> torch.Tensor:
        """
        Returns foreground probabilities in shape [B, H, W], float32.
        Supports logits shaped [B,1,H,W] or [B,2,H,W].
        """
        if pred_logits.ndim != 4:
            raise ValueError(f"Expected pred_logits [B,C,H,W], got {tuple(pred_logits.shape)}")

        B, C, H, W = pred_logits.shape
        if C == 1:
            # Binary logit -> sigmoid prob
            probs = torch.sigmoid(pred_logits[:, 0, ...])
        elif C == 2:
            # BG/FG logits -> softmax -> take FG channel 1
            probs = torch.softmax(pred_logits, dim=1)[:, 1, ...]
        else:
            raise ValueError(f"Expected C=1 or C=2 for binary segmentation, got C={C}")

        return probs.to(dtype=torch.float32)

    @staticmethod
    def _extract_target_fg(target: torch.Tensor) -> torch.Tensor:
        """
        Returns foreground target mask in shape [B, H, W], bool.
        Supports target shaped [B,H,W] with {0,1} or [B,2,H,W] one-hot.
        """
        if target.ndim == 3:
            # [B,H,W]
            tgt = target
        elif target.ndim == 4 and target.shape[1] == 2:
            # [B,2,H,W] one-hot -> take FG channel 1
            tgt = target[:, 1, ...]
        elif target.ndim == 4 and target.shape[1] == 1:
            tgt = target[:, 0, ...]
        else:
            raise ValueError(f"Unsupported target shape: {tuple(target.shape)}")

        # Convert to bool (foreground)
        if tgt.dtype != torch.bool:
            tgt = tgt > 0.5
        return tgt

    @torch.no_grad()
    def update(self, pred_logits: torch.Tensor, target: torch.Tensor):
        """
        Incrementally updates AUPRC/AUROC + collapse stats + MCC* counts.
        """
        probs_fg = self._extract_probs_fg(pred_logits)  # [B,H,W], float32
        tgt_fg = self._extract_target_fg(target)        # [B,H,W], bool

        # Flatten
        p = probs_fg.reshape(-1)
        y = tgt_fg.reshape(-1)

        # Update torchmetrics (keep on device)
        # Torchmetrics expects preds float in [0,1] and target int/bool
        self.auprc.update(p, y)
        self.auroc.update(p, y)

        # Collapse stats @ 0.5
        pred_pos = (p >= 0.5).sum().item()
        total = p.numel()
        self._pred_pos_at_05 += int(pred_pos)
        self._total_pixels += int(total)

        # MCC* counts per threshold (accumulate on CPU)
        # We compute counts on-device then move small K-vectors to CPU.
        thresholds = self.mcc_thresholds.to(p.device)

        # Vectorized across thresholds (K x N) can be large if N huge;
        # but K=9 so it’s usually fine. If needed, you can switch to a loop.
        preds_k = p.unsqueeze(0) >= thresholds.unsqueeze(1)  # [K,N] bool
        y_k = y.unsqueeze(0)                                 # [1,N] bool (broadcast)

        tp = (preds_k & y_k).sum(dim=1)
        fp = (preds_k & ~y_k).sum(dim=1)
        tn = (~preds_k & ~y_k).sum(dim=1)
        fn = (~preds_k & y_k).sum(dim=1)

        # Accumulate on CPU int64
        self._tp += tp.to("cpu", dtype=torch.int64)
        self._fp += fp.to("cpu", dtype=torch.int64)
        self._tn += tn.to("cpu", dtype=torch.int64)
        self._fn += fn.to("cpu", dtype=torch.int64)

    @staticmethod
    def _mcc_from_counts(tp, fp, tn, fn, eps: float = 1e-12) -> torch.Tensor:
        tp = tp.to(torch.float64)
        fp = fp.to(torch.float64)
        tn = tn.to(torch.float64)
        fn = fn.to(torch.float64)
        num = tp * tn - fp * fn
        den = torch.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn) + eps)
        return (num / den).to(torch.float32)

    @staticmethod
    def _extract_probs_from_probs_fg(probs_fg: torch.Tensor) -> torch.Tensor:
        """
        probs_fg: [B,H,W] or [B,1,H,W] float in [0,1]
        returns:  [B,H,W] float32 in [0,1]
        """
        if probs_fg.ndim == 4 and probs_fg.shape[1] == 1:
            probs_fg = probs_fg[:, 0, ...]
        if probs_fg.ndim != 3:
            raise ValueError(f"Expected probs_fg [B,H,W] (or [B,1,H,W]), got {tuple(probs_fg.shape)}")
        return probs_fg.to(dtype=torch.float32).clamp(0.0, 1.0)

    @torch.no_grad()
    def update_from_probs_fg(self, probs_fg: torch.Tensor, target: torch.Tensor):
        """
        Update metrics using already-computed FG probabilities (e.g., from TTA).
        probs_fg: [B,H,W] float in [0,1]
        target:   [B,2,H,W] one-hot OR [B,1,H,W] OR [B,H,W]
        """
        probs_fg = self._extract_probs_from_probs_fg(probs_fg)  # [B,H,W] float32
        tgt_fg = self._extract_target_fg(target)                # [B,H,W] bool

        p = probs_fg.reshape(-1)
        y = tgt_fg.reshape(-1)

        self.auprc.update(p, y)
        self.auroc.update(p, y)

        pred_pos = (p >= 0.5).sum().item()
        total = p.numel()
        self._pred_pos_at_05 += int(pred_pos)
        self._total_pixels += int(total)

        thresholds = self.mcc_thresholds.to(p.device)
        preds_k = p.unsqueeze(0) >= thresholds.unsqueeze(1)  # [K,N]
        y_k = y.unsqueeze(0)                                 # [1,N]

        tp = (preds_k & y_k).sum(dim=1)
        fp = (preds_k & ~y_k).sum(dim=1)
        tn = (~preds_k & ~y_k).sum(dim=1)
        fn = (~preds_k & y_k).sum(dim=1)

        self._tp += tp.to("cpu", dtype=torch.int64)
        self._fp += fp.to("cpu", dtype=torch.int64)
        self._tn += tn.to("cpu", dtype=torch.int64)
        self._fn += fn.to("cpu", dtype=torch.int64)

    def compute_and_reset(self, health: TrainingHealthTracker = None):
        """
        Returns:
          None if collapsed or invalid metrics (with printed reason),
          else dict with:
            val_auprc, val_auroc, val_mcc_star, fg_prevalence_at_05
        """
        # No samples processed
        if self._total_pixels == 0:
            print("[VAL GUARDRAIL] No pixels processed in validation epoch.")
            self.reset()
            return None

        fg_prev = self._pred_pos_at_05 / float(self._total_pixels)

        # Guardrail 1: FG prevalence collapse
        if fg_prev < self.collapse_low:
            print(
                f"[VAL GUARDRAIL] COLLAPSE DETECTED — "
                f"FG prevalence @0.5 too LOW: {fg_prev:.6f} "
                f"(threshold {self.collapse_low:.6f})"
            )
            if health is not None:health.mark_val_collapsed();
            self.reset()
            return None

        if fg_prev > self.collapse_high:
            print(
                f"[VAL GUARDRAIL] COLLAPSE DETECTED — "
                f"FG prevalence @0.5 too HIGH: {fg_prev:.6f} "
                f"(threshold {self.collapse_high:.6f})"
            )
            if health is not None:health.mark_val_collapsed();
            self.reset()
            return None

        # Compute streaming AUPRC / AUROC
        auprc_t = self.auprc.compute().detach().cpu()
        auroc_t = self.auroc.compute().detach().cpu()

        # Guardrail 2: NaN / Inf AUPRC
        if (auprc_t.numel() == 0) or (not torch.isfinite(auprc_t).all()):
            print(f"[VAL GUARDRAIL] INVALID METRIC — AUPRC is NaN/Inf: {auprc_t}")
            if health is not None:
                health.mark_val_invalid_metrics()
            self.reset()
            return None

        # Guardrail 3: NaN / Inf AUROC
        if (auroc_t.numel() == 0) or (not torch.isfinite(auroc_t).all()):
            print(f"[VAL GUARDRAIL] INVALID METRIC — AUROC is NaN/Inf: {auroc_t}")
            if health is not None:
                health.mark_val_invalid_metrics()
            self.reset()
            return None

        val_auprc = float(auprc_t.item())
        val_auroc = float(auroc_t.item())

        # Compute MCC* over thresholds
        mccs = self._mcc_from_counts(self._tp, self._fp, self._tn, self._fn)

        # Guardrail 4: NaN / Inf MCC*
        if (mccs.numel() == 0) or (not torch.isfinite(mccs).all()):
            print(f"[VAL GUARDRAIL] INVALID METRIC — MCC* contains NaN/Inf: {mccs}")
            if health is not None:
                health.mark_val_invalid_metrics()
            self.reset()
            return None

        val_mcc_star = float(torch.max(mccs).item())

        # Final defensive check
        if not (math.isfinite(val_auprc) and math.isfinite(val_auroc) and math.isfinite(val_mcc_star)):
            print(f"[VAL GUARDRAIL] INVALID METRIC — AUPRC={val_auprc}, AUROC={val_auroc}, MCC*={val_mcc_star}")
            if health is not None:
                health.mark_val_invalid_metrics()
            self.reset()
            return None

        out = {
            "val_auprc": val_auprc,
            "val_auroc": val_auroc,
            "val_mcc_star": val_mcc_star,
            "fg_prevalence_at_05": float(fg_prev),
        }

        self.reset()
        return out

In [ ]:
class RunningWeightedMetric:
    def __init__(self):
        self.reset()

    def reset(self):
        self.cumulative_score = 0.0
        self.sample_count = 0

    def update(self, batch_score_sum, batch_sample_count):
        """
        batch_score_sum: The sum of metrics in the batch (e.g., sum of Dices)
        batch_sample_count: The number of valid samples in that batch (e.g., count of tumor images)
        """
        self.cumulative_score += batch_score_sum
        self.sample_count += batch_sample_count

    def get_average(self):
        return self.cumulative_score / self.sample_count if self.sample_count > 0 else 0.0

In [ ]:
class ProstateCancerDataset(Dataset):
    def __init__(
        self,
        cancer_image_dir,
        cancer_mask_dir,
        not_cancer_image_dir,
        not_cancer_mask_dir,
        mean=None,
        std=None,
        compute_stats=False,
        stats_sample_size=None,  # e.g. 10_000 patches to speed up estimation
    ):
        self.cancer_image_dir = cancer_image_dir
        self.cancer_mask_dir = cancer_mask_dir
        self.not_cancer_image_dir = not_cancer_image_dir
        self.not_cancer_mask_dir = not_cancer_mask_dir

        self.patient_ids = []

        # Combine and store file paths and labels
        self.image_paths = []
        self.mask_paths = []
        self.labels = []

        # Compile the regex once for efficiency
        patient_id_pattern = re.compile(r'PATIENT_(\d+)_')

        # --- Process CANCER images ---
        cancer_images = []
        if os.path.isdir(self.cancer_image_dir):
            cancer_images = [f for f in os.listdir(cancer_image_dir) if f.lower().endswith('.png')]

        for img_name in cancer_images:
            mask_path = os.path.join(self.cancer_mask_dir, img_name)
            match = patient_id_pattern.search(img_name)
            if os.path.isfile(mask_path) and match:
                self.image_paths.append(os.path.join(self.cancer_image_dir, img_name))
                self.mask_paths.append(mask_path)
                self.labels.append(1)
                self.patient_ids.append(match.group(1))

        # --- Process NOT_CANCER images ---
        not_cancer_images = []
        if os.path.isdir(self.not_cancer_image_dir):
            not_cancer_images = [f for f in os.listdir(not_cancer_image_dir) if f.lower().endswith('.png')]

        for img_name in not_cancer_images:
            mask_path = os.path.join(self.not_cancer_mask_dir, img_name)
            match = patient_id_pattern.search(img_name)
            if os.path.isfile(mask_path) and match:
                self.image_paths.append(os.path.join(self.not_cancer_image_dir, img_name))
                self.mask_paths.append(mask_path)
                self.labels.append(0)
                self.patient_ids.append(match.group(1))

        # ------------------------------------------------------------------
        #  NEW: dataset-specific mean/std (after stain normalization)
        # ------------------------------------------------------------------
        if mean is not None and std is not None:
            # Use externally provided stats (recommended for VAL / TEST)
            self.mean = mean if isinstance(mean, list) else list(mean)
            self.std = std if isinstance(std, list) else list(std)
            print(f"[ProstateCancerDataset] Using provided mean/std: "
                  f"mean={self.mean}, std={self.std}")
        elif compute_stats:
            # Compute stats from this dataset (recommended for TRAIN set)
            print("[ProstateCancerDataset] Computing dataset-specific mean/std...")
            self.mean, self.std = self._compute_dataset_mean_std(
                max_samples=stats_sample_size
            )
            print(f"[ProstateCancerDataset] Computed mean: {self.mean}")
            print(f"[ProstateCancerDataset] Computed std:  {self.std}")
        else:
            # Fallback: ImageNet stats (old behavior, but less ideal scientifically)
            self.mean = [0.485, 0.456, 0.406]
            self.std = [0.229, 0.224, 0.225]
            print("[ProstateCancerDataset] Using ImageNet mean/std "
                  "(no dataset-specific stats requested).")

        # --- Base Transformation (Applied to ALL data) ---
        # NOTE: Albumentations.Normalize expects mean/std in [0,1] scale when
        # max_pixel_value=255.0 (default). We computed them in that scale.
        self.base_transform = A.Compose([
            A.Resize(224, 224, interpolation=cv2.INTER_LINEAR),  # Specify interpolation
            A.Normalize(mean=self.mean, std=self.std),
            ToTensorV2(),  # Handles image scaling & channel order
        ])

    def _compute_dataset_mean_std(self, max_samples=None):
        """
        Compute per-channel mean and std over this dataset's images.

        Args:
            max_samples (int or None): if set, randomly sample up to this many
                images to estimate stats (for speed). If None, use all images.

        Returns:
            mean (list of 3 floats), std (list of 3 floats) in [0,1] scale.
        """
        # If dataset is empty, fall back to ImageNet to avoid crashes
        if len(self.image_paths) == 0:
            print("[ProstateCancerDataset] WARNING: No images found; "
                  "falling back to ImageNet stats.")
            return [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

        # Decide which indices to use
        indices = np.arange(len(self.image_paths))
        if max_samples is not None and max_samples < len(indices):
            np.random.shuffle(indices)
            indices = indices[:max_samples]

        n_pixels_total = 0
        channel_sum = np.zeros(3, dtype=np.float64)
        channel_sum_sq = np.zeros(3, dtype=np.float64)

        for idx in indices:
            img_path = self.image_paths[idx]
            img = cv2.imread(img_path, cv2.IMREAD_COLOR)
            if img is None:
                continue  # skip broken images

            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = img.astype(np.float32) / 255.0  # scale to [0,1]

            # Flatten to (N, 3)
            img_flat = img.reshape(-1, 3)

            n_pixels = img_flat.shape[0]
            n_pixels_total += n_pixels

            channel_sum += img_flat.sum(axis=0)
            channel_sum_sq += (img_flat ** 2).sum(axis=0)

        if n_pixels_total == 0:
            print("[ProstateCancerDataset] WARNING: Failed to load any pixels "
                  "while computing stats; falling back to ImageNet stats.")
            return [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

        mean = channel_sum / n_pixels_total
        var = (channel_sum_sq / n_pixels_total) - mean ** 2
        std = np.sqrt(np.maximum(var, 1e-12))

        return mean.tolist(), std.tolist()

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]

        try:
            image = cv2.imread(img_path, cv2.IMREAD_COLOR)
            if image is None: raise IOError("cv2.imread failed for image")
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB) # Convert to RGB for consistency if needed downstream

            mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            if mask is None: raise IOError("cv2.imread failed for mask")

        except Exception as e:
            print(f"Error loading image/mask: {img_path} / {mask_path} - {e}")
            # Return None tuple, handled by collate_fn
            return None, None

        # Create Two-Channel Mask (One-Hot Encode) before transform
        mask = mask.astype(np.uint8)
        two_channel_mask = np.zeros((mask.shape[0], mask.shape[1], 2), dtype=np.float32)
        two_channel_mask[mask == 0, 0] = 1.0  # Background channel
        two_channel_mask[mask != 0, 1] = 1.0  # Cancer channel

        # Apply the base transformations
        try:
            # Pass mask correctly (shape H, W, C)
            augmented = self.base_transform(image=image, mask=two_channel_mask)
            final_image = augmented['image'] # Shape (C, H, W), FloatTensor, Normalized
            final_mask = augmented['mask']   # Shape (C, H, W), FloatTensor, Values 0.0 or 1.0

            # Ensure mask shape is (2, 224, 224)
            if final_mask.shape[0] != 2:
                 resized_mask = cv2.resize(mask, (224, 224), interpolation=cv2.INTER_NEAREST)
                 two_channel_mask_resized = np.zeros((224, 224, 2), dtype=np.float32)
                 two_channel_mask_resized[resized_mask == 0, 0] = 1.0
                 two_channel_mask_resized[resized_mask != 0, 1] = 1.0
                 final_mask = torch.from_numpy(two_channel_mask_resized).permute(2, 0, 1) # HWC -> CHW

            # Final check on mask shape
            if final_mask.shape != (2, 224, 224):
                 raise ValueError(f"Final mask shape is incorrect: {final_mask.shape}")


        except Exception as e:
             print(f"Error applying transform to {os.path.basename(img_path)}: {e}")
             return None, None # Return None tuple on transform error


        return final_image, final_mask

    def get_class_counts(self):
        """Returns the counts of cancer (1) and not-cancer (0) samples."""
        counts = np.bincount(self.labels)
        not_cancer_count = counts[0] if len(counts) > 0 else 0
        cancer_count = counts[1] if len(counts) > 1 else 0
        return {'CANCER': cancer_count, 'NOT_CANCER': not_cancer_count}

print("Dataset definition complete.")

In [ ]:
def build_mined_weights(dataset, oversample_factor: float = 4.0):
    """
    dataset can be ProstateCancerDataset or Subset(ProstateCancerDataset).
    Returns a float tensor of per-sample weights.
    """
    if isinstance(dataset, Subset):
        base = dataset.dataset
        idxs = dataset.indices
        paths = [base.image_paths[i] for i in idxs]
    else:
        paths = dataset.image_paths

    w = []
    for p in paths:
        name = os.path.basename(p)
        w.append(oversample_factor if name.startswith("MINED_") else 1.0)
    return torch.tensor(w, dtype=torch.double)

In [ ]:
# --- Helper Functions ---
def get_formatted_datetime_string():
  now = datetime.now()
  return now.strftime("%d_%m_%Y_%H_%M_%S")

In [ ]:
# --- Seeding ---
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    # Keep benchmark=False for determinism if needed, but True might be faster
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print("Configuration complete.")

In [ ]:
def clear_gpu():
    print("Clearing GPU cache...")
    if torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        except Exception as e:
            # Optional: only print if something really went wrong
            print(f"[clear_gpu] Warning: {e}")
    print("GPU cache cleared.")

In [ ]:
class EarlyStopping:
    """
    Early stops the training and saves only the single best model checkpoint,
    deleting the previous best to conserve disk space.
    """
    def __init__(self, patience=5, verbose=True, delta=0.0001, output_best_model_path='best_model.pth'):
        """
        Args:
            output_best_model_path (str): The fixed path where the single best model will be saved.
        """
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.early_stop = False
        self.best_score = None
        self.delta = delta
        self.output_best_model_path = output_best_model_path # The fixed target path for saving
        self._current_best_checkpoint_on_disk_path = None # Tracks the path of the *actual* best file on disk

        os.makedirs(os.path.dirname(self.output_best_model_path), exist_ok=True)

    def set_initial_best_checkpoint_path(self, path):
        if path and os.path.exists(path):
            self._current_best_checkpoint_on_disk_path = path
            if self.verbose:
                print(f"EarlyStopping: Initial best checkpoint set to {os.path.basename(path)}")
        else:
            self._current_best_checkpoint_on_disk_path = None
            if self.verbose:
                print("EarlyStopping: No initial best checkpoint path provided/found.")

    def __call__(self, score, model, optimizer, epoch, val_loss, val_auprc, val_mcc_star):
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model, optimizer, epoch, val_auprc, val_mcc_star, score)
            return True

        improvement_detected = False
        if score > self.best_score + self.delta:
            improvement_detected = True

        if improvement_detected:
            previous_best_score_for_log = self.best_score
            self.best_score = score
            self.save_checkpoint(val_loss, model, optimizer, epoch, val_auprc, val_mcc_star, score, previous_best_score_for_log)
            self.counter = 0
        else:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience} (Best score: {self.best_score:.6f})')
            if self.counter >= self.patience:
                self.early_stop = True

        return improvement_detected

    def save_checkpoint(self, val_loss, model, optimizer, epoch, val_auprc, val_mcc_star, score, previous_best_score_for_log=None):
        if self.verbose:
            if previous_best_score_for_log is None:
                print(f'Initial best score: {score:.6f}. Saving model...')

        model_to_save = model._orig_mod if hasattr(model, '_orig_mod') else model

        save_dict = {
            'epoch': epoch,
            'model_state_dict': model_to_save.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
            'best_val_score': score, # This is AUPRC
            'val_auprc': val_auprc,
            'val_mcc_star': val_mcc_star,
            'is_compiled': hasattr(model, '_orig_mod')
        }

        try:
            torch.save(save_dict, self.output_best_model_path)
            if self.verbose:
                print(f"  New best model saved to: {self.output_best_model_path}")

            self._current_best_checkpoint_on_disk_path = self.output_best_model_path

        except Exception as e:
            print(f"Error saving best model checkpoint to {self.output_best_model_path}: {e}")

In [ ]:
def create_stratified_subset(full_dataset, ratio, split_name="Unknown"):
    """
    Creates a scientifically robust, stratified random subsample of a dataset.

    This function performs a two-level stratification:
    1. It groups all patches by their source patient.
    2. Within each patient, it further groups patches by class (Cancer/Not Cancer).
    3. It then samples the specified ratio from each of these sub-groups.

    This ensures the final subset precisely preserves the class proportions
    within each patient from the original full dataset.
    """
    print(f"  Creating a {ratio:.0%} two-level stratified subsample for {split_name.upper()} (by patient and class)...")

    indices_by_patient_and_class = defaultdict(lambda: defaultdict(list))
    for i in range(len(full_dataset)):
        # These attributes must exist on the dataset object
        patient_id = full_dataset.patient_ids[i]
        label = full_dataset.labels[i]
        indices_by_patient_and_class[patient_id][label].append(i)

    print(f"    Found {len(indices_by_patient_and_class)} unique patients in the {split_name} split.")

    subset_indices = []
    generator = torch.Generator().manual_seed(SEED)

    for patient_id, class_groups in indices_by_patient_and_class.items():
        for label, indices in class_groups.items():
            num_to_sample = int(np.ceil(len(indices) * ratio))
            shuffled_indices = torch.randperm(len(indices), generator=generator).tolist()
            sampled_local_indices = shuffled_indices[:num_to_sample]
            subset_indices.extend([indices[i] for i in sampled_local_indices])

    random.shuffle(subset_indices)

    # Calculate and print the "after" counts for verification
    subset_labels = [full_dataset.labels[i] for i in subset_indices]
    if subset_labels:
        subset_counts = np.bincount(subset_labels)
        not_cancer_count = subset_counts[0] if len(subset_counts) > 0 else 0
        cancer_count = subset_counts[1] if len(subset_counts) > 1 else 0
    else:
        not_cancer_count, cancer_count = 0, 0

    print(f"    {split_name.title()} subset class counts -> CANCER: {cancer_count}, NOT_CANCER: {not_cancer_count}")

    return Subset(full_dataset, subset_indices)

In [ ]:
def _setup_precision():
    """
    Decide autocast dtype + scaler and toggle TF32 based on model type and GPU.
    Returns: (amp_dtype, scaler)
    """
    if IS_DPT:
        # Strict FP32 for DPT (stability). Also keep TF32 OFF to ensure full FP32 math.
        try:
            torch.set_float32_matmul_precision('high')  # disable TF32 optimizations
        except Exception:
            pass
        return torch.float32, None

    # Non-DPT path: prefer bf16, else fp16+GradScaler. Enable TF32 for any remaining FP32 ops.
    try:
        torch.set_float32_matmul_precision('high')  # TF32 on Ampere/Ada; harmless elsewhere
    except Exception:
        pass

    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    if use_bf16:
        return torch.bfloat16, None
    else:
        return torch.float16, torch.cuda.amp.GradScaler()

print("Defining training and validation functions...")

In [ ]:
def train_model(model, optimizer, dataloader, device, current_epoch, loss_fn, health):
    model.train()
    if hasattr(optimizer, "train"):
        optimizer.train()

    amp_dtype, scaler = _setup_precision()

    # Trackers for Batch-Size Bias Correction
    tracker_loss  = RunningWeightedMetric()

    pbar = tqdm(dataloader, desc=f"Train E{current_epoch+1}", leave=False)

    for batch_data in pbar:
        # --- Guardrail 1: DataLoader yielded None ---
        if batch_data is None:
            health.train_skip("dataloader_none_batch")
            continue

        # --- Guardrail 2: Unpack defensively ---
        try:
            images, masks = batch_data
        except Exception as e:
            health.train_skip("unpack_failed")
            continue

        # --- Guardrail 3: None images/masks ---
        if images is None or masks is None:
            health.train_skip("images_or_masks_none")
            continue

        # --- Guardrail 4: Empty batch ---
        try:
            bsz = images.size(0)
        except Exception as e:
            health.train_skip("images_no_batch_dim")
            continue

        if bsz == 0:
            health.train_skip("batch_size_zero")
            continue

        images = images.to(device, non_blocking=True)
        masks  = masks.to(device, non_blocking=True)  # keep your current behavior (no .float())

        optimizer.zero_grad(set_to_none=True)

        # --- Forward + Loss (with output guardrails) ---
        ctx = autocast_ctx(images, amp_dtype)
        with ctx:
            outputs_raw = model(images)

            # Guardrail 5: compiled/model may return tuple/list
            if isinstance(outputs_raw, (tuple, list)):
                if len(outputs_raw) == 0:
                    health.train_skip("model_empty_tuple")
                    continue
                outputs = outputs_raw[0]
            else:
                outputs = outputs_raw

            # Guardrail 6: output must be tensor
            if not isinstance(outputs, torch.Tensor):
                health.train_skip("model_output_not_tensor")
                continue

            # Guardrail 7: shape mismatch
            if hasattr(masks, "shape") and outputs.shape != masks.shape:
                health.train_skip("shape_mismatch")
                continue

            loss = loss_fn(outputs, masks)

        # Guardrail 8: NaN/Inf loss
        if not torch.isfinite(loss):
            health.train_naninf_loss()
            health.train_skip("naninf_loss")
            optimizer.zero_grad(set_to_none=True)
            continue

        # --- Backward/Step (AMP-safe) ---
        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        tracker_loss.update(loss.item() * bsz, bsz)

        pbar.set_postfix(
            loss=f"{tracker_loss.get_average():.4f}"
            )

    return tracker_loss.get_average()

In [ ]:
def validate_model(model, optimizer, dataloader, device, loss_fn, health):
    model.eval()
    if hasattr(optimizer, "eval"):
        optimizer.eval()

    # Precision setup (match training)
    amp_dtype, _ = _setup_precision()

    tracker = AdvancedMetricTracker(device=device, metric_bins=8192)

    running_loss = 0.0
    num_samples_processed = 0

    pbar = tqdm(dataloader, desc="Validate", leave=False)

    with torch.inference_mode():
        for batch_idx, batch_data in enumerate(pbar):
            # --- Guardrail 1: DataLoader yielded None ---
            if batch_data is None:
                health.val_skip("dataloader_none_batch")
                continue

            # --- Guardrail 2: Unpack defensively ---
            try:
                images, masks = batch_data
            except Exception as e:
                health.val_skip("unpack_failed")
                continue

            # --- Guardrail 3: None images/masks ---
            if images is None or masks is None:
                health.val_skip("images_or_masks_none")
                continue

            # --- Guardrail 4: Empty batch / invalid batch size ---
            try:
                bsz = images.size(0)
            except Exception as e:
                health.val_skip("images_no_batch_dim")
                continue

            if bsz == 0:
                health.val_skip("batch_size_zero")
                continue

            images = images.to(device, non_blocking=True)
            masks  = masks.to(device, non_blocking=True)  # keep current behavior (no forced .float())

            # --- Forward + loss with AMP ---
            ctx = autocast_ctx(images, amp_dtype)
            with ctx:
                outputs_raw = model(images)

                # --- Guardrail 5: Handle tuple/list output (compiled models, etc.) ---
                if isinstance(outputs_raw, (tuple, list)):
                    if len(outputs_raw) == 0:
                        health.val_skip("model_empty_tuple")
                        continue
                    outputs = outputs_raw[0]
                else:
                    outputs = outputs_raw

                # --- Guardrail 6: Output must be a tensor ---
                if not isinstance(outputs, torch.Tensor):
                    health.val_skip("model_output_not_tensor")
                    continue

                # --- Guardrail 7: Shape mismatch ---
                if outputs.shape != masks.shape:
                  health.val_skip("shape_mismatch")
                  continue

                loss = loss_fn(outputs, masks)

            # --- Guardrail 8: NaN/Inf loss ---
            if not torch.isfinite(loss):
                health.val_naninf_loss()
                health.val_skip("naninf_loss")
                continue

            # --- Accumulate loss (sample-weighted) ---
            batch_loss = loss.item()
            running_loss += batch_loss * bsz
            num_samples_processed += bsz

            # --- Update global metrics tracker (current behavior) ---
            if USE_TTA_FOR_EVAL:
                # Metrics computed with TTA probabilities (coherent with threshold tuning)
                probs_fg_tta = predict_with_tta(model, images, device, amp_dtype)  # [B,H,W]
                tracker.update_from_probs_fg(probs_fg_tta, masks)
            else:
                # Fallback: metrics from single-pass logits
                tracker.update(outputs, masks)

            # Pbar update (loss only; metrics at the end)
            if num_samples_processed > 0:
                pbar.set_postfix(
                    loss=f"{batch_loss:.4f}",
                    avg_loss=f"{running_loss / num_samples_processed:.4f}",
                )

    # End of epoch aggregation
    if num_samples_processed == 0:
      health.val_skip("no_samples_processed")
      return 0.0, None

    epoch_loss = running_loss / num_samples_processed
    epoch_results = tracker.compute_and_reset(health=health)

    return epoch_loss, epoch_results

In [ ]:
# --- Update create_email_body to include AUC ---
def create_email_body(checkpoint_path, encoder, architecture):
    body = f'Training {architecture} finished.\n\nCheckpoint Path: {checkpoint_path}\n\n--- ENCODER: {encoder} ---\n'
    return body

def send_email(subject, body, sender, recipients, password):
    msg = MIMEText(body); msg['Subject'] = subject; msg['From'] = sender; msg['To'] = ', '.join(recipients)
    try:
        with smtplib.SMTP_SSL('smtp.gmail.com', 465) as smtp_server: smtp_server.login(sender, password); smtp_server.sendmail(sender, recipients, msg.as_string())
        print("Email sent successfully!")
    except Exception as e: print(f"Error sending email: {e}")

In [ ]:
@torch.no_grad()
def predict_with_tta(model, images, device, amp_dtype):
    model.eval()
    images = images.to(device, non_blocking=True)

    # safe autocast context
    use_cuda_amp = images.is_cuda and (amp_dtype is not torch.float32)
    ctx = torch.amp.autocast("cuda", dtype=amp_dtype) if use_cuda_amp else contextlib.nullcontext()

    tta_probs = []

    # ---- 1) Original ----
    with ctx:
        out = model(images)
        if isinstance(out, (tuple, list)):
            out = out[0]
        probs = torch.softmax(out, dim=1)[:, 1]
        tta_probs.append(probs)

    # ---- 2) Horizontal flip ----
    with ctx:
        imgs_h = torch.flip(images, dims=[3])
        out = model(imgs_h)
        if isinstance(out, (tuple, list)):
            out = out[0]
        probs = torch.softmax(out, dim=1)[:, 1]
        probs = torch.flip(probs, dims=[2])
        tta_probs.append(probs)

    # ---- 3) Vertical flip ----
    with ctx:
        imgs_v = torch.flip(images, dims=[2])
        out = model(imgs_v)
        if isinstance(out, (tuple, list)):
            out = out[0]
        probs = torch.softmax(out, dim=1)[:, 1]
        probs = torch.flip(probs, dims=[1])
        tta_probs.append(probs)

    return torch.stack(tta_probs, dim=0).mean(dim=0)  # [B,H,W]

In [ ]:
def make_single_model_predict_proba_fn(model):
    amp_dtype, _ = _setup_precision()
    @torch.no_grad()
    def _predict(images):
        probs = predict_with_tta(model, images, device, amp_dtype)  # [B,H,W]
        return probs
    return _predict

In [ ]:
import torch

@torch.no_grad()
def find_threshold_max_tnr_subject_tpr(
    dataloader,
    predict_proba_fn,          # callable(images) -> probs_fg [B,H,W] in [0,1]
    device,
    tpr_target: float = 0.95,
    bins: int = 4096,          # more bins = more precision; 1024/2048 is often enough
    eps: float = 1e-12,
):
    """
    Streaming, memory-safe threshold selection:
      maximize TNR subject to TPR >= tpr_target.
    Uses histograms of probs for positives and negatives.

    Returns:
      best_threshold, summary_dict
    """
    pos_hist = torch.zeros(bins, dtype=torch.int64)
    neg_hist = torch.zeros(bins, dtype=torch.int64)

    total_pos = 0
    total_neg = 0

    for batch in dataloader:
        if batch is None:
            continue
        images, masks = batch
        if images is None or masks is None:
            continue

        images = images.to(device, non_blocking=True)
        masks  = masks.to(device, non_blocking=True)

        # Ground truth FG mask: [B,H,W] bool
        if masks.ndim == 4 and masks.shape[1] == 2:
            y = (masks[:, 1] > 0.5)
        elif masks.ndim == 4 and masks.shape[1] == 1:
            y = (masks[:, 0] > 0.5)
        elif masks.ndim == 3:
            y = (masks > 0.5)
        else:
            raise ValueError(f"Unsupported mask shape: {tuple(masks.shape)}")

        probs = predict_proba_fn(images)  # [B,H,W] float in [0,1]
        probs = probs.clamp(0, 1).reshape(-1).detach().to("cpu")
        y = y.reshape(-1).detach().to("cpu")

        # bucketize into [0..bins-1]
        idx = torch.clamp((probs * (bins - 1)).long(), 0, bins - 1)

        pos_idx = idx[y]
        neg_idx = idx[~y]

        if pos_idx.numel():
            pos_hist += torch.bincount(pos_idx, minlength=bins)
            total_pos += int(pos_idx.numel())
        if neg_idx.numel():
            neg_hist += torch.bincount(neg_idx, minlength=bins)
            total_neg += int(neg_idx.numel())

    if total_pos == 0 or total_neg == 0:
        # Degenerate validation set; return a safe default
        return 0.5, {
            "status": "degenerate",
            "total_pos": total_pos,
            "total_neg": total_neg,
        }

    # For threshold t, we predict positive if prob >= t.
    # Using bins: threshold at bin k corresponds to prob >= k/(bins-1)
    # TP(k) = sum_{i>=k} pos_hist[i]
    # FP(k) = sum_{i>=k} neg_hist[i]
    pos_cum = torch.cumsum(pos_hist.flip(0), dim=0).flip(0)
    neg_cum = torch.cumsum(neg_hist.flip(0), dim=0).flip(0)

    TP = pos_cum.to(torch.float64)
    FP = neg_cum.to(torch.float64)
    FN = (total_pos - TP)
    TN = (total_neg - FP)

    TPR = TP / (total_pos + eps)
    TNR = TN / (total_neg + eps)

    # Feasible thresholds
    feasible = TPR >= tpr_target
    if feasible.any():
        # maximize TNR among feasible
        best_idx = torch.argmax(torch.where(feasible, TNR, torch.tensor(-1.0, dtype=TNR.dtype)))
        status = "ok"
    else:
        # If target unattainable, pick max TPR (min FN), tie-break max TNR
        max_tpr = torch.max(TPR)
        cand = (TPR == max_tpr)
        best_idx = torch.argmax(torch.where(cand, TNR, torch.tensor(-1.0, dtype=TNR.dtype)))
        status = "unattainable_tpr_target"

    best_idx = int(best_idx.item())
    best_threshold = best_idx / float(bins - 1)

    summary = {
        "status": status,
        "tpr_target": float(tpr_target),
        "best_threshold": float(best_threshold),
        "TPR": float(TPR[best_idx].item()),
        "TNR": float(TNR[best_idx].item()),
        "FPR": float((1.0 - TNR[best_idx]).item()),
        "total_pos": int(total_pos),
        "total_neg": int(total_neg),
        "bins": int(bins),
    }
    return best_threshold, summary

In [ ]:
# --- 12. Aim & Ngrok Setup ---
print("Setting up Aim repository...")
# --- Aim Setup ---
from aim import Run
import subprocess
import aim
if not os.path.exists(AIM_REPO_PATH):
  print(f"Aim repository not found at {AIM_REPO_PATH}. Initializing...")
  os.makedirs(AIM_REPO_PATH, exist_ok=True)
  try:
    repo = aim.Repo.init(AIM_REPO_PATH);
    print(f"Aim repository initialized at: {repo.path}")
  except Exception as e:
    print(f"An unexpected error occurred during Aim repo setup: {e}")
else:
  print(f"Using existing Aim repository at: {AIM_REPO_PATH}")
print("Aim/Ngrok setup complete.")

In [ ]:
def save_metadata(optimal_threshold, best_val_score, checkpoint, encoder, architecture, metadata_best_path, val_loss, val_mcc, val_auroc, thr_info):
  # --- SAVE THE OPTIMAL THRESHOLD ---
  meta_filename = os.path.join(METADATA_DIR,os.path.basename(metadata_best_path))
  meta_filename = meta_filename.replace(".pth", "_meta.json")

  metadata = {
    "optimal_threshold_optimized": optimal_threshold,
    "thr_FPR": thr_info["FPR"],
    "thr_TPR": thr_info["TPR"],
    "thr_TNR": thr_info["TNR"],
    "best_val_auprc_score": best_val_score,
    "val_loss": val_loss,
    "val_mcc": val_mcc,
    "val_auroc": val_auroc,
    "best_model_epoch": checkpoint.get('epoch', '?'),
    "checkpoint_path": metadata_best_path,
    "encoder": encoder,
    "architecture": architecture,
    "Learning_rate": BASE_LEARNING_RATE,
    "Encoder_LR_Factor": ENCODER_LR_FACTOR,
    "Weight_Decay": WEIGHT_DECAY,
    "Batch_Size": BATCH_SIZE,
    "Num_Epochs": NUM_EPOCHS,
    "Workers": WORKERS,
    "Seed": SEED,
    "DATASET_ZIP_DIR": DATASET_ZIP_DIR,
    "Dropout": DECODER_DROPOUT,
    "patience": PATIENCE,
    "Optimizer": OPTIMIZER_NAME,
    "Loss": "BCEDiceHybrid",
    "ALPHA_BCE": ALPHA_BCE,
    "BETA_DICE_BG": BETA_DICE_BG,
    "GAMMA_DICE_FG": GAMMA_DICE_FG
    }

  try:
      with open(meta_filename, 'w') as f:
          json.dump(metadata, f, indent=4)
      print(f"Saved optimal threshold and metadata to: {meta_filename}")
  except Exception as e:
    print(f"Error saving metadata file {meta_filename}: {e}")

In [ ]:
def get_model(architecture, encoder,validation=False):

  if validation:
    aux_params=None
  else:
    aux_params=dict(dropout=DECODER_DROPOUT, classes=2)

  encoder_weights = None if validation else "imagenet"

  if architecture=="SWIN":
    model = smp.Unet(
    encoder_name=encoder,
    encoder_weights=encoder_weights,
    in_channels=3,
    classes=2,
    activation=None,
    decoder_attention_type=None,
    aux_params=aux_params)
  elif architecture=="DEEPLABV3PLUS":
    model = smp.DeepLabV3Plus(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  elif architecture=="INCEPTIONRESNETV2":
    model = smp.Unet(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  elif architecture=="DPT":
    model = smp.DPT(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        decoder_readout='ignore',
        aux_params=aux_params)
  elif architecture=="UNET++":
    model = smp.UnetPlusPlus(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  elif architecture=="FPN":
    model = smp.FPN(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  elif architecture=="SEGFORMER":
    model = smp.Segformer(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  elif architecture=="MANET":
    model = smp.MAnet(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  elif architecture=="UPERNET":
    model = smp.UPerNet(
        encoder_name=encoder,
        encoder_weights=encoder_weights,
        in_channels=3,
        classes=2,
        activation=None,
        decoder_attention_type=None,
        aux_params=aux_params)
  else:
    raise ValueError(f"Unknown architecture: {architecture}")

  return model

In [ ]:
def load_checkpoint_for_resume(model, optimizer, early_stopping, checkpoint_path, device):
    """
    Loads a training checkpoint (if provided) and returns the starting epoch index
    for the outer training loop. It also informs EarlyStopping about the initial best path.
    """
    start_epoch = 0

    if not checkpoint_path:
        print("No resume checkpoint provided: training from scratch.")
        early_stopping.set_initial_best_checkpoint_path(None)
        return start_epoch

    checkpoint_path = checkpoint_path.strip()
    if checkpoint_path == "":
        print("Empty resume checkpoint string: training from scratch.")
        early_stopping.set_initial_best_checkpoint_path(None)
        return start_epoch

    if not os.path.exists(checkpoint_path):
        print(f"Resume checkpoint not found at {checkpoint_path}. Training from scratch.")
        early_stopping.set_initial_best_checkpoint_path(None)
        return start_epoch

    print(f"\n*** Resuming from checkpoint: {checkpoint_path}")
    chkpt = torch.load(checkpoint_path, map_location=device)

    state_dict = chkpt.get("model_state_dict", None)
    if state_dict is None:
        print("Checkpoint missing 'model_state_dict'. Training from scratch.")
        early_stopping.set_initial_best_checkpoint_path(None)
        return start_epoch

    if hasattr(model, "_orig_mod"):
        model._orig_mod.load_state_dict(state_dict)
    else:
        model.load_state_dict(state_dict)

    if "optimizer_state_dict" in chkpt:
        try:
            optimizer.load_state_dict(chkpt["optimizer_state_dict"])
            print("Optimizer state loaded from checkpoint.")
        except Exception as e:
            print(f"Warning: could not load optimizer state: {e}")

    best_score = chkpt.get("best_val_score", None)
    if best_score is not None:
        early_stopping.best_score = best_score
        # previous_best_score is not needed for EarlyStopping logic, only for print messages
        # early_stopping.previous_best_score = best_score # Removed as it's not strictly part of ES logic
        early_stopping.counter = 0
        print(f"Loaded EarlyStopping best_val_score = {best_score:.6f}")


    # This is the "best checkpoint on disk" initially.
    early_stopping.set_initial_best_checkpoint_path(checkpoint_path)

    last_epoch = int(chkpt.get("epoch", 0))
    start_epoch = last_epoch
    print(f"Last finished epoch in checkpoint: {last_epoch}. "
          f"Next epoch will be {last_epoch + 1}.\n")

    return start_epoch

In [ ]:
# ==============================================================================
# --- 13. Main Training Loop ---
# ==============================================================================
print(f"\n{'='*25} Starting Main Training Process {'='*25}")

fold_zip_filename = f'MASTER_SET_1.zip'
fold_zip_path = os.path.join(DATASET_ZIP_DIR, fold_zip_filename)

# --- Extract Dataset ---
print(f"Extracting Fold...")
if not os.path.exists(fold_zip_path):
  print(f"Zip not found: {fold_zip_path}. Skip.");
try:
    if os.path.exists(base_data_dir):
      shutil.rmtree(base_data_dir)
    os.makedirs(base_data_dir, exist_ok=True);

    with zipfile.ZipFile(fold_zip_path,'r') as z:
      z.extractall(base_data_dir)

    print("Extracted. Verifying...");

    if not os.path.isdir(train_cancer_image_dir) or not os.listdir(train_cancer_image_dir):
      raise RuntimeError("Verify failed")

    print("Verified.")

except Exception as e:
  print(f"Extract Err: {e}. Skip.");
  clear_gpu();

In [ ]:
# --- DataLoaders (with optional stratified subsampling) ---
print("\nCreating DataLoaders...")

sample_size = int(STATS_SAMPLE_SIZE) if STATS_SAMPLE_SIZE is not None else None
mean_to_be_used = MEAN if (MEAN is not None and len(MEAN) > 0) else None
std_to_be_used = STD if (STD is not None and len(STD) > 0) else None

try:
    # Always instantiate the full datasets first
    # 1) TRAIN: compute dataset-specific stats on full TRAIN
    full_train_ds = ProstateCancerDataset(
        train_cancer_image_dir,
        train_cancer_mask_dir,
        train_not_cancer_image_dir,
        train_not_cancer_mask_dir,
        compute_stats=False,
        stats_sample_size=sample_size,
        mean = mean_to_be_used,
        std = std_to_be_used
    )

    train_mean = full_train_ds.mean
    train_std  = full_train_ds.std

    # 2) VAL: reuse TRAIN stats (no compute_stats here!)
    full_val_ds = ProstateCancerDataset(
        val_cancer_image_dir,
        val_cancer_mask_dir,
        val_not_cancer_image_dir,
        val_not_cancer_mask_dir,
        mean=train_mean,
        std=train_std,
    )

    # --- NEW: Print "Before" counts ---
    train_counts_before = full_train_ds.get_class_counts()
    val_counts_before = full_val_ds.get_class_counts()

    print("\n--- Full Dataset Class Counts (Before Subsampling) ---")
    print(f" TRAIN:      CANCER={train_counts_before['CANCER']}, NOT_CANCER={train_counts_before['NOT_CANCER']}")
    print(f" VALIDATION: CANCER={val_counts_before['CANCER']}, NOT_CANCER={val_counts_before['NOT_CANCER']}")
    print('-'*50)

    def collate_fn(batch):
      batch = list(filter(lambda x: x is not None and x[0] is not None, batch))
      return torch.utils.data.dataloader.default_collate(batch) if batch else None

    # --- Stratified Subsampling Logic ---
    if USE_SUBSET and SUBSET_RATIO < 1.0:
        print(f"Subsampling enabled. Using {SUBSET_RATIO:.0%} of the data for TRAIN and VALIDATION.")

        # --- MODIFICATION: Call the single, correct function for both datasets ---
        train_ds = create_stratified_subset(full_train_ds, SUBSET_RATIO, split_name="training")
        val_ds = create_stratified_subset(full_val_ds, SUBSET_RATIO, split_name="validation")

        print(f"\n--- Subset DS Lengths ---\n Train: {len(train_ds)}, Val: {len(val_ds)}\n{'-'*30}")

    else:
        print("Using full datasets for training and validation.")
        train_ds = full_train_ds
        val_ds = full_val_ds

    weights = build_mined_weights(train_ds, oversample_factor=OVERSAMPLE_FACTOR)
    sampler = WeightedRandomSampler(weights=weights, num_samples=len(weights), replacement=True)

    print("Oversample factor:", OVERSAMPLE_FACTOR)
    print("Num mined in train_ds:", int((weights > 1.0).sum().item()), "/", len(weights))


    train_loader = DataLoader(train_ds,
                              batch_size=BATCH_SIZE,
                              sampler=sampler,          # ✅ sampler instead of shuffle
                              shuffle=False,            # ✅ must be False with sampler
                              num_workers=WORKERS,
                              pin_memory=True,
                              drop_last=True,
                              persistent_workers=WORKERS>0,
                              prefetch_factor=2 if WORKERS>0 else None,
                              collate_fn=collate_fn)


    val_loader = DataLoader(val_ds,
                            BATCH_SIZE,
                            shuffle=False,
                            num_workers=WORKERS,
                            pin_memory=True,
                            persistent_workers=WORKERS>0,
                            prefetch_factor=2 if WORKERS>0 else None,
                            collate_fn=collate_fn)

    print("DataLoaders created successfully.")

except Exception as e:
  print(f"DataLoader Err: {e}. Skip.")
  clear_gpu()

In [ ]:
def define_optimizer():
  if OPTIMIZER_NAME == "AdamWScheduleFree":
    optimizer = schedulefree.AdamWScheduleFree(
        model.parameters(),
        lr=BASE_LEARNING_RATE,
        weight_decay=WEIGHT_DECAY)
  elif OPTIMIZER_NAME == "AdamW":
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=BASE_LEARNING_RATE,
        weight_decay=WEIGHT_DECAY)
  else:
    raise ValueError(f"Unknown optimizer: {OPTIMIZER_NAME}")
  return optimizer

In [ ]:
# # # # # The new, refactored lists
# LIST_ARCH = ['SWIN',
#              'DEEPLABV3PLUS',
#              'UNET++',
#              'FPN',
#              'SEGFORMER',
#              'MANET',
#              'DPT',
#              'UPERNET'
#              ]

# LIST_ENCODER = [
#      'tu-swin_large_patch4_window7_224.ms_in22k_ft_in1k', # For Unet (acting as our new Swin model)
#      'tu-resnest101e',                                    # For DEEPLABV3PLUS
#      'efficientnet-b7',                                   # For UNET++
#      'senet154',                                          # For FPN
#      'mit_b5',                                            # For SEGFORMER
#      'resnet152',                                         # For MANET
#      'tu-vit_large_patch16_224.augreg_in21k_ft_in1k',     # For DPT
#      'tu-hiera_large_224'                                 # For UPerNet
#  ]


# # LIST_CHECKPOINT = [
# #      '',                                                 # For Unet (acting as our new Swin model)
# #      '',                                                 # For DEEPLABV3PLUS
# #      '',                                                 # For UNET++
# #      '',                                                 # For FPN
# #      '',                                                 # For SEGFORMER
# #      '',                                                 # For MANET
# #      '',                                                 # For DPT
# #      ''                                                  # For UPerNet
# #  ]

In [ ]:
LIST_ARCH = ['SWIN']

LIST_ENCODER = ['tu-swin_large_patch4_window7_224.ms_in22k_ft_in1k']

LIST_CHECKPOINT = ['']

for architecture, encoder, resume_checkpoint_path in zip(LIST_ARCH, LIST_ENCODER, LIST_CHECKPOINT):

  best_val_score_for_architecture_auprc = None
  best_val_score_for_architecture_mcc = None
  best_val_loss_for_architecture_auroc = None
  best_val_loss_for_architecture_loss = None

  health = TrainingHealthTracker(name=f"{architecture}_{encoder}")
  BASE_LEARNING_RATE,WEIGHT_DECAY  = get_learning_rate(architecture)
  if architecture=="DPT":
    IS_DPT=True
  else:
    IS_DPT=False

  # --- Init Aim Run ---
  experiment_name = f"{IDENTIFIER}_{architecture}_{encoder}_{get_formatted_datetime_string()}"
  print(f"Init Aim: {experiment_name}")
  run = None
  try:
      run = Run(experiment=experiment_name, repo=AIM_REPO_PATH)
      run["hparams"] = {
          "base_learning_rate": BASE_LEARNING_RATE, "encoder_lr_factor": ENCODER_LR_FACTOR, "weight_decay": WEIGHT_DECAY,
          "batch_size": BATCH_SIZE, "num_epochs": NUM_EPOCHS,"workers": WORKERS, "seed": SEED,"label": experiment_name,
          "optimizer": OPTIMIZER_NAME,"loss": "BCEDiceHybridLossPaper","model": f"{architecture}_{encoder}",
          "encoder_weights": "IMAGENET", "patience": PATIENCE,
          "train_len":len(train_ds),"val_len":len(val_ds),
          "loss_alpha_bce": ALPHA_BCE, "loss_beta_dice_bg": BETA_DICE_BG, "loss_gamma_dice_fg": GAMMA_DICE_FG,
      }
      print("Aim run initialized.")
  except Exception as e:
    print(f"Aim Init Err: {e}.")

  print(f"Architecture: {architecture}")
  # --- Initialize Model, Optimizer, Early Stopping ---
  print("Initializing SMP Model, Optimizer, ES...")

  # --- Instantiate SMP Model ---
  try:
    model = get_model(architecture = architecture, encoder = encoder, validation = False)
    model.to(device);
  except Exception as e:
    print(f"Model init err: {e}")
    raise ValueError(f"Unknown architecture: {architecture}")

  # --- Compile Model ---
  try:
    model = torch.compile(model);
    print("Model compiled.")
  except Exception as e:
    print(f"Compile failed: {e}.")

  print(f"Model->{device}")

  optimizer = define_optimizer()

  print(f"Optimizer initialized with {OPTIMIZER_NAME}")

  output_best_model_path_for_this_run = os.path.join(CHECKPOINT_PATH, f'BEST_MODEL_{experiment_name}.pth')

  # Initialize EarlyStopping with this fixed path
  early_stopping = EarlyStopping(
      patience=int(PATIENCE),
      verbose=True,
      delta=0.0001,
      output_best_model_path=output_best_model_path_for_this_run)

  full_resume_checkpoint_path = os.path.join(CHECKPOINT_PATH, resume_checkpoint_path) if resume_checkpoint_path else None

  start_epoch = load_checkpoint_for_resume(
      model=model,
      optimizer=optimizer,
      early_stopping=early_stopping,
      checkpoint_path=full_resume_checkpoint_path,
      device=device)

  if early_stopping._current_best_checkpoint_on_disk_path:
      # If we successfully loaded a resume checkpoint, that's our initial best.
      metadata_best_path = early_stopping._current_best_checkpoint_on_disk_path
  else:
      # If training from scratch, this will be the first best model saved.
      metadata_best_path = output_best_model_path_for_this_run


  print(f"Starting Training For {architecture} from epoch {start_epoch + 1}...")

  loss_fn = BCEDiceHybridLossPaper(
      alpha=ALPHA_BCE,
      beta=BETA_DICE_BG,
      gamma=GAMMA_DICE_FG
      )
  print(
      f"Using BCE+Dice Hybrid Loss "
      f"(alpha={ALPHA_BCE}, beta={BETA_DICE_BG}, gamma={GAMMA_DICE_FG})"
      )

  training_successful = True
  for epoch in range(start_epoch, NUM_EPOCHS):
      health.reset_epoch()
      current_epoch_num = epoch + 1
      epoch_start = time.time()

      try:
        t_loss = train_model(model, optimizer, train_loader, device, epoch, loss_fn, health)
      except Exception as e:
        print(f"\nTrain Err E{current_epoch_num}:{e}")
        training_successful=False
        break

      try:
          v_loss, v_results = validate_model(model, optimizer, val_loader, device, loss_fn, health)

          # 1. Check for Model Collapse (if tracker returned None)
          if v_results is None:
              if health.epoch["val_collapsed"] == 1:
                  reason = "COLLAPSE"
              elif health.epoch["val_invalid_metrics"] == 1:
                  reason = "INVALID_METRICS"
              else:
                  reason = "UNKNOWN"
              print(f"\n[Epoch {current_epoch_num}] Validation failed ({reason}). Skipping checkpoint.")
              health.log_epoch(current_epoch_num)
              continue

          # 2. Unpack the robust metrics
          val_auprc = v_results['val_auprc']
          val_auroc = v_results['val_auroc']
          val_mcc_star = v_results['val_mcc_star']

          # DECISION RULE:
          # Use AUPRC as the score to maximize.
          # (Since AUPRC is highly correlated with segmentation quality in imbalance)
          score = val_auprc
      except Exception as e:
          print(f"\nVal Err E{current_epoch_num}: {e}")
          import traceback
          traceback.print_exc()
          training_successful = False
          break

      # --- Logging ---
      epoch_dur=time.time()-epoch_start
      mins,secs=divmod(epoch_dur,60)
      print(
        f"\nE{current_epoch_num}/{NUM_EPOCHS} [{int(mins):02d}m{int(secs):02d}s] "
        f"Tr L:{t_loss:.4f}|"
        f"Val AUPRC:{val_auprc:.4f} AUROC:{val_auroc:.4f} MCC*:{val_mcc_star:.4f}"
        )

      if run:
          try:
              run.track(t_loss, 'loss', epoch=current_epoch_num, context={"subset": "train"})
              run.track(val_auprc,  'auprc', epoch=current_epoch_num, context={"subset": "val"})
              run.track(val_auroc,  'auroc', epoch=current_epoch_num, context={"subset": "val"})
              run.track(val_mcc_star,   'mcc',  epoch=current_epoch_num, context={"subset": "val"})
          except Exception as e:
              print(f"Aim Log Err: {e}")

      try:
          # We use AUPRC as the primary score for Early Stopping
          improvement_detected = early_stopping(score, model, optimizer, current_epoch_num, v_loss, val_auprc, val_mcc_star)

          if improvement_detected:
              # If EarlyStopping saved a NEW best, then metadata_best_path should reflect this fixed path.
              # early_stopping._current_best_checkpoint_on_disk_path will already be updated inside ES.
              best_val_score_for_architecture_auprc = val_auprc
              best_val_score_for_architecture_mcc = val_mcc_star
              best_val_loss_for_architecture_auroc = val_auroc
              best_val_loss_for_architecture_loss = v_loss
              metadata_best_path = early_stopping.output_best_model_path

              print(f"  >>> New Best Model! (AUPRC: {val_auprc:.4f})")
      except Exception as e:
          print(f"ES/Save Err: {e}")

      if not UNLEASHED and early_stopping.early_stop:
        print(f"Early stopping E{current_epoch_num}.")
        break

      print(f"Epoch {epoch} completed.")
      health.log_epoch(current_epoch_num)


  # --- Post-Training for Fold ---
  if not training_successful:
    print(f"Train loop stopped early for {architecture}.")
  else:
    print(f"\nTrain loop finished for {architecture}.")
    clear_gpu()

  if SIMULATION_ONLY:
    sys.exit(0)
  # --- Load Best Model (based on Val Loss) ---
  print("Loading best model for threshold tuning...")

  # The EarlyStopping instance now holds the definitive path to the best checkpoint on disk.
  final_best_checkpoint_path = early_stopping._current_best_checkpoint_on_disk_path

  if final_best_checkpoint_path and os.path.exists(final_best_checkpoint_path):
      best_model_path = final_best_checkpoint_path
  else:
      best_model_path = full_resume_checkpoint_path

  try:
    if os.path.exists(best_model_path):
        chkpt=torch.load(best_model_path,map_location=device)

        # --- Instantiate SMP Model ---
        try:
          model_test = get_model(architecture = architecture, encoder = encoder, validation = True)
          model_test.to(device)
        except Exception as e:
          print(f"Model init err: {e}")
          raise ValueError(f"Unknown architecture: {architecture}")

        print("Loading state dict...");
        model_test.load_state_dict(chkpt['model_state_dict'],strict=False)
        print("Loaded.");
        is_comp=chkpt.get('is_compiled',False)

        if is_comp:
          print("Compiling test model...")
          try:
            model_test=torch.compile(model_test)
            print("Compiled.")
          except Exception as e:
            print(f"Compile fail: {e}")

        #full val loader
        val_loader_tune = DataLoader(
          full_val_ds, BATCH_SIZE, shuffle=False,
          num_workers=WORKERS, pin_memory=True,
          persistent_workers=WORKERS>0,
          prefetch_factor=2 if WORKERS>0 else None,
          collate_fn=collate_fn
          )

        #optimization
        predict_fn = make_single_model_predict_proba_fn(model_test)
        thr, info = find_threshold_max_tnr_subject_tpr(
            dataloader=val_loader_tune,
            predict_proba_fn=predict_fn,
            device=device,
            tpr_target=SENSITIVITY_TARGET,
            bins=2048,   # usually enough; 4096 if you want
        )
        print(thr, info)

        save_metadata(
            optimal_threshold=thr,
            best_val_score=best_val_score_for_architecture_auprc,
            checkpoint=chkpt,
            encoder=encoder,
            architecture=architecture,
            metadata_best_path=best_model_path,
            val_loss=best_val_loss_for_architecture_loss,
            val_mcc=best_val_score_for_architecture_mcc,
            val_auroc=best_val_loss_for_architecture_auroc,
            thr_info=info)


        print("Emailing...");
        subject = f"Training {architecture} finished"
        body=create_email_body(metadata_best_path, encoder, architecture);
        send_email(f"Finished: {experiment_name}",body,sender,recipients,password)
    else:
      print(f"Best model not found: {best_model_path}. Skip test.")

  except Exception as e:
    print(f"Test/Visu Err: {e}")
  import traceback;
  traceback.print_exc()

  if run:
    run.close()
    print("Aim run closed.")

  print(f"====== METRIC STABILITY FOR {architecture} ======")
  health.log_run()

  clear_gpu();
  print(f"\n{'='*20} Finished Fold {architecture} {'='*20}")
  time.sleep(3)


# --- Final Cleanup ---
print("\nAll folds processed.")